Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# これまでの進捗（簡易版）

| Ver. | 内容 | CV スコア | LB スコア |
| ---- | ---- | ---- | ---- |
| 1 | パイプライン構築 | - | 0.57033 |
| 2 | CVの導入（Hold-out検証） | 0.8411 | 0.83389 |
| 4 | CVの検討（KFold） | 0.8327 | 0.82837 |
| 5 | テキストクレンジング | 0.8349 | 0.82163 |

In [1]:
import keras
import keras_nlp

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

import kagglehub

In [2]:
import kagglehub

path = kagglehub.competition_download("nlp-getting-started")

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/nlp-getting-started


In [3]:
import os

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))

# データの概要

In [4]:
train.head(10)

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
5,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed...,1
6,10,NaN,NaN,#flood #disaster Heavy rain causes flash flood...,1
7,13,NaN,NaN,I'm on top of the hill and I can see a fire in...,1
8,14,NaN,NaN,There's an emergency evacuation happening now ...,1
9,15,NaN,NaN,I'm afraid that the tornado is coming to our a...,1


In [5]:
test.head(10)

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan
5,12,NaN,NaN,We're shaking...It's an earthquake
6,21,NaN,NaN,They'd probably still show more life than Arse...
7,22,NaN,NaN,Hey! How are you?
8,27,NaN,NaN,What a nice hat?
9,29,NaN,NaN,Fuck off!


In [6]:
sample_submission.head()

,id,target
0,0,0
1,2,0
2,3,0
3,9,0
4,11,0


In [7]:
train.shape

(7613, 5)

# 関数の定義

In [8]:
import html

def clean_text(df):
    df_clean = df.copy()

    # 1. HTML特殊文字の復元 (例: &amp; -> &)
    df_clean["text"] = df_clean["text"].apply(html.unescape)

    # 2. keyword がある場合は text の先頭に結合する (例: "earthquake: Our Deeds are...")
    # NaN の場合は空文字にして処理
    keywords = df_clean["keyword"].fillna("").astype(str)

    # keyword が空でないものだけ "keyword: " を頭につけて結合
    df_clean["text"] = keywords.apply(
        lambda k: f"{k}: " if k else ""
    ) + df_clean["text"]

    return df_clean


# train と test に適用
train_cleaned = clean_text(train)
test_cleaned = clean_text(test)

# 実行

In [9]:
%%time

# 1. パラメータ設定
FOLDS = 5
EPOCHS = 2
BATCH_SIZE = 16    # <- 32 から 16 に変更（メモリ消費を半分に抑える）
LR = 1e-5
PRESET = "deberta_v3_base_en"


# 2. StratifiedKFold の準備
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

# OOF（学習用データの検証予測）と Test（テストデータの予測）の格納用配列
oof_predictions = np.zeros(len(train_cleaned))
test_predictions = np.zeros(len(test_cleaned))


# 3. K-Fold ループ
for fold, (train_idx, val_idx) in enumerate(
    skf.split(train_cleaned, train_cleaned["target"])
):
    print(f"\n========== Fold {fold + 1} / {FOLDS} ==========")

    # 前処理済み（clean_text 適用済み）データを指定
    X_train, y_train = (
        train_cleaned["text"].values[train_idx],
        train_cleaned["target"].values[train_idx],
    )
    X_val, y_val = (
        train_cleaned["text"].values[val_idx],
        train_cleaned["target"].values[val_idx],
    )

    # モデルと前処理の構築（Fold ごとにリセット）
    preprocessor = keras_nlp.models.TextClassifierPreprocessor.from_preset(
        PRESET, sequence_length=160
    )
    
    model = keras_nlp.models.TextClassifier.from_preset(
        PRESET, preprocessor=preprocessor, num_classes=2
    )

    model.compile(
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=keras.optimizers.AdamW(learning_rate=LR),
        metrics=["accuracy"],
    )

    # Fold ごとにベストモデルを保存
    checkpoint_path = f"best_model_fold{fold}.keras"
    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
    )

    # 学習
    model.fit(
        x=X_train,
        y=y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=[checkpoint_cb],
        verbose=1,
    )

    # 各 Fold のベストモデルを読み込んで検証＆推論
    best_model = keras.models.load_model(checkpoint_path)

    # Validation データの予測（OOF）
    val_preds = best_model.predict(X_val)
    oof_predictions[val_idx] = val_preds.argmax(axis=-1)

    # Test データへの予測（前処理済みの test_cleaned を使用）
    test_preds = best_model.predict(test_cleaned["text"].values)
    test_predictions += test_preds.argmax(axis=-1) / FOLDS


# 4. 全体の CV スコア算出（OOF Accuracy）
oof_accuracy = (oof_predictions == train_cleaned["target"].values).mean()
print(f"\n==========================================")
print(f"Overall OOF CV Accuracy: {oof_accuracy:.4f}")
print(f"==========================================")


# 5. 提出ファイルの作成（5つのモデルのアンサンブル）
test["target"] = (test_predictions >= 0.5).astype(int)
test[["id", "target"]].to_csv("submission.csv", index=False)
print("submission.csv を出力しました！")


========== Fold 1 / 5 ==========


I0000 00:00:1790857562.776023      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790857562.778829      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/2


I0000 00:00:1790857673.809227      72 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


381/381 ━━━━━━━━━━━━━━━━━━━━ 501s 1s/step - accuracy: 0.7415 - loss: 0.5415 - val_accuracy: 0.8207 - val_loss: 0.4198
Epoch 2/2
381/381 ━━━━━━━━━━━━━━━━━━━━ 341s 895ms/step - accuracy: 0.8381 - loss: 0.3958 - val_accuracy: 0.8404 - val_loss: 0.3800


/usr/local/lib/python3.12/dist-packages/keras/src/saving/serialization_lib.py:749: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 406 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


48/48 ━━━━━━━━━━━━━━━━━━━━ 46s 809ms/step
102/102 ━━━━━━━━━━━━━━━━━━━━ 65s 640ms/step

========== Fold 2 / 5 ==========
Epoch 1/2
381/381 ━━━━━━━━━━━━━━━━━━━━ 494s 1s/step - accuracy: 0.7002 - loss: 0.5627 - val_accuracy: 0.8214 - val_loss: 0.4140
Epoch 2/2
381/381 ━━━━━━━━━━━━━━━━━━━━ 344s 902ms/step - accuracy: 0.8363 - loss: 0.3957 - val_accuracy: 0.8391 - val_loss: 0.3914
48/48 ━━━━━━━━━━━━━━━━━━━━ 41s 740ms/step
102/102 ━━━━━━━━━━━━━━━━━━━━ 65s 639ms/step

========== Fold 3 / 5 ==========
Epoch 1/2
381/381 ━━━━━━━━━━━━━━━━━━━━ 501s 1s/step - accuracy: 0.7407 - loss: 0.5336 - val_accuracy: 0.8168 - val_loss: 0.4253
Epoch 2/2
381/381 ━━━━━━━━━━━━━━━━━━━━ 334s 876ms/step - accuracy: 0.8409 - loss: 0.3834 - val_accuracy: 0.8083 - val_loss: 0.4534
48/48 ━━━━━━━━━━━━━━━━━━━━ 42s 750ms/step
102/102 ━━━━━━━━━━━━━━━━━━━━ 66s 643ms/step

========== Fold 4 / 5 ==========
Epoch 1/2
381/381 ━━━━━━━━━━━━━━━━━━━━ 510s 1s/step - accuracy: 0.7155 - loss: 0.5555 - val_accuracy: 0.8430 - val_loss: 0

## 簡易実験用

In [10]:
# %%time

# # -------------------------------------------------------------------
# # 簡易実験フラグ（DEBUG = True で1,000件のみ高速実行 / False で全データ）
# # -------------------------------------------------------------------
# DEBUG = False
# SAMPLE_SIZE = 1000

# if DEBUG:
#     train_df = train_cleaned.sample(n=SAMPLE_SIZE, random_state=42).reset_index(
#         drop=True
#     )
#     print(f"【 DEBUG モード】データ数を {len(train_df)} 件に絞って高速実行します。")
# else:
#     train_df = train_cleaned.copy()


# # 1. データの分割（Hold-out）
# X_train, X_val, y_train, y_val = train_test_split(
#     train_df["text"].values,
#     train_df["target"].values,
#     test_size=0.2,
#     # random_state=0,
#     stratify=train_df["target"].values,
# )


# # 2. モデルの準備
# PRESET = "deberta_v3_base_en"

# preprocessor = keras_nlp.models.TextClassifierPreprocessor.from_preset(
#     PRESET, sequence_length=160
# )
# model = keras_nlp.models.TextClassifier.from_preset(
#     PRESET, preprocessor=preprocessor, num_classes=2
# )


# # 3. コンパイル
# model.compile(
#     loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#     optimizer=keras.optimizers.AdamW(learning_rate=1e-5),
#     metrics=["accuracy"],
# )


# # 4. Checkpoint 設定
# checkpoint_path = "debug_best_model.keras"
# checkpoint_cb = keras.callbacks.ModelCheckpoint(
#     filepath=checkpoint_path,
#     monitor="val_accuracy",
#     save_best_only=True,
#     mode="max",
# )


# # 5. 高速学習
# history = model.fit(
#     x=X_train,
#     y=y_train,
#     validation_data=(X_val, y_val),
#     batch_size=32,
#     epochs=2,
#     callbacks=[checkpoint_cb],
#     verbose=1,
# )


# # 6. ベストモデルの評価
# best_model = keras.models.load_model(checkpoint_path)
# best_val_loss, best_val_acc = best_model.evaluate(X_val, y_val, verbose=0)
# print(
#     f"\n[Quick Hold-out] Best Validation Accuracy: {best_val_acc:.4f} (DEBUG={DEBUG})"
# )

※デバッグモードありだと 2 分以内には終わる。

DeBERTa v3

[Quick Hold-out] Best Validation Accuracy: 0.8450 (DEBUG=False)
CPU times: user 4min 27s, sys: 48 s, total: 5min 15s
Wall time: 15min 46s

In [11]:
# %%time

# # 1. データの分割（Validation データを作成）
# X_train, X_val, y_train, y_val = train_test_split(
#     train["text"].values,
#     train["target"].values,
#     test_size=0.2,
#     random_state=0,
#     stratify=train["target"].values,
# )

# # 2. モデルと前処理の準備
# preset = "distil_bert_base_en_uncased"
# preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(
#     preset, sequence_length=160
# )
# model = keras_nlp.models.DistilBertClassifier.from_preset(
#     preset, preprocessor=preprocessor, num_classes=2
# )

# # 3. コンパイル（純正の keras.optimizers.AdamW を使用）
# # 学習率 2e-5 を指定、デフォルトは大きすぎる
# optimizer = keras.optimizers.AdamW(learning_rate=2e-5)
# loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# model.compile(
#     loss=loss,
#     optimizer=optimizer,
#     metrics=["accuracy"],
# )

# # 4. 学習（ CV 精度の確認）
# history = model.fit(
#     x=X_train,
#     y=y_train,
#     validation_data=(X_val, y_val),
#     batch_size=32,
#     epochs=2,
# )

# # 5. 推論と提出ファイルの作成
# predictions = model.predict(test["text"].values)
# test["target"] = predictions.argmax(axis=-1)

# test[["id", "target"]].to_csv("submission.csv", index=False)
# print("submission.csv を出力しました！")